In [44]:
# ============================================================
# AURORA ASSIGNEE PREDICTOR — FULL PIPELINE V5
# With TF-IDF + Keyword Features + Reporter Encoding
# ============================================================

import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier
import joblib

In [45]:
# ============================================================
# CONFIG
# ============================================================

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, "assignee_model_aurora_v5.joblib")
LABEL_ENCODER_PATH = os.path.join(MODEL_DIR, "label_encoder_aurora_v5.joblib")
TFIDF_PATH = os.path.join(MODEL_DIR, "tfidf_vectorizer_aurora_v5.joblib")

USE_MINILM = False  # turn ON later if needed

PROJECTS = {
    "Aurora": {
        "issues":  r"PM_Kaggle_dataset\AgileScrumSprintVelocityDataSet\Agile Scrum Dataset\Finalized Datasets for Aurora Project\Aurora Issues 554.csv",
        "summary": r"PM_Kaggle_dataset\AgileScrumSprintVelocityDataSet\Agile Scrum Dataset\Finalized Datasets for Aurora Project\Aurora Issues summery 568.csv",
        "sprints": r"PM_Kaggle_dataset\AgileScrumSprintVelocityDataSet\Agile Scrum Dataset\Finalized Datasets for Aurora Project\Aurora Sprints 41.csv",
    }
}

ALLOWED_ASSIGNEES = [
    "Maxim Khutornenko", "Bill Farner", "Kevin Sweeney",
    "Mark Chu-Carroll", "Zameer Manji", "Joshua Cohen",
    "Brian Wickman", "David McLaughlin",
]

KEYWORDS = [
    "scheduler","mesos","protobuf","agent","quota","api",
    "frontend","backend","refactor","migration","resource",
    "tracking","build","testing","framework","executor"
]


In [46]:
# ============================================================
# LOAD + MERGE AURORA DATA
# ============================================================

def load_aurora():
    p = PROJECTS["Aurora"]

    df_issues  = pd.read_csv(p["issues"])
    df_summary = pd.read_csv(p["summary"])
    df_sprints = pd.read_csv(p["sprints"])

    print("\n--- Loaded Raw Files ---")
    print("Issues:", df_issues.shape)
    print("Summary:", df_summary.shape)
    print("Sprints:", df_sprints.shape)

    # Merge Issues ↔ Summary
    df = df_issues.merge(
        df_summary,
        left_on="key",
        right_on="issueKey",
        how="left",
        suffixes=("", "_summary"),
    )

    # Merge Issues ↔ Sprints
    df = df.merge(
        df_sprints,
        left_on="sprint",
        right_on="sprintId",
        how="left",
        suffixes=("", "_sprint"),
    )

    print("Merged:", df.shape)
    return df

In [47]:
# ============================================================
# BUILD FEATURE MATRIX (TF-IDF + NUMERIC)
# ============================================================

def build_feature_matrix(df, tfidf=None, fit_tfidf=False, emb_model=None):
    # Ensure required text columns exist
    if "summary" not in df.columns:
        df["summary"] = ""

    if "description" not in df.columns:
        df["description"] = ""

    df["summary"] = df["summary"].fillna("")
    df["description"] = df["description"].fillna("")

    # Keyword flags
    for kw in KEYWORDS:
        df[f"kw_{kw}"] = df["summary"].str.contains(kw, case=False, na=False).astype(int)

    # Summary length
    df["summary_len"] = df["summary"].apply(lambda x: len(x.split()))

    # Reporter one-hot encoding (if exists)
    if "reporter" in df.columns:
        df = pd.get_dummies(df, columns=["reporter"], prefix="rpt")

    # Build list of numeric columns
    numeric_cols = [c for c in df.columns if c.startswith("kw_") or c.startswith("rpt_")]
    numeric_cols.append("summary_len")

    # Clean dtypes (bool → int)
    for col in numeric_cols:
        if df[col].dtype == bool:
            df[col] = df[col].astype(int)

    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

    # TF-IDF
    if fit_tfidf:
        print("\nFitting TF-IDF...")
        tfidf = TfidfVectorizer(max_features=600, stop_words="english")
        X_tfidf = tfidf.fit_transform(df["summary"])
    else:
        print("\nTransforming with existing TF-IDF...")
        X_tfidf = tfidf.transform(df["summary"])

    # Build numeric sparse matrix
    X_num = csr_matrix(df[numeric_cols].astype(float).values)

    # Combine
    X_sparse = hstack([X_tfidf, X_num]).tocsr()

    # MiniLM embedding (optional)
    if USE_MINILM:
        emb_list = []
        for text in df["summary"].values:
            emb_list.append(emb_model.encode(str(text)))
        X_emb = np.vstack(emb_list)
    else:
        X_emb = np.zeros((len(df), 1))  # placeholder

    X_final = np.hstack([X_sparse.toarray(), X_emb])
    print("X_final shape:", X_final.shape)

    return X_final, tfidf, numeric_cols

In [48]:
# ============================================================
# TRAINING PIPELINE
# ============================================================

def train_pipeline():
    print("\n🚀 Training Aurora Assignee Predictor...")

    df = load_aurora()

    # Clean + whitelist
    df = df[df["assignee"].notna()].copy()
    df = df[df["assignee"].isin(ALLOWED_ASSIGNEES)].copy()

    print("\nAfter whitelist:", df.shape)
    print(df["assignee"].value_counts())

    # Oversample
    max_count = df["assignee"].value_counts().max()
    df_balanced = (
        df.groupby("assignee")
          .apply(lambda x: x.sample(max_count, replace=True, random_state=42))
          .reset_index(drop=True)
    )

    print("\nAfter oversampling:")
    print(df_balanced["assignee"].value_counts())

    # Encode labels
    le = LabelEncoder()
    y = le.fit_transform(df_balanced["assignee"])

    # Build features
    X_final, tfidf, numeric_cols = build_feature_matrix(
        df_balanced,
        tfidf=None,
        fit_tfidf=True
    )
    joblib.dump(numeric_cols, os.path.join(MODEL_DIR, "numeric_cols_aurora_v5.joblib"))

    # Save tfidf + label encoder
    joblib.dump(tfidf, TFIDF_PATH)
    joblib.dump(le, LABEL_ENCODER_PATH)

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_final, y, test_size=0.20, random_state=42, stratify=y
    )

    # Train model
    model = XGBClassifier(
        n_estimators=500,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.7,
        eval_metric="mlogloss",
        tree_method="hist",
    )

    print("\nTraining XGBoost...")
    model.fit(X_train, y_train)
    joblib.dump(model, MODEL_PATH)

    # Evaluate
    print("\n📊 Classification Report:")
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=le.classes_))

    # Top-K
    probs = model.predict_proba(X_test)
    top1 = accuracy_score(y_test, y_pred)
    top3 = np.mean([y_test[i] in np.argsort(probs[i])[-3:] for i in range(len(y_test))])
    top5 = np.mean([y_test[i] in np.argsort(probs[i])[-5:] for i in range(len(y_test))])

    print("\nTop-1 Accuracy:", round(top1, 3))
    print("Top-3 Accuracy:", round(top3, 3))
    print("Top-5 Accuracy:", round(top5, 3))

    # Sample predictions
    print("\nSample predictions (first 10):")
    for i in range(100):
        true = le.inverse_transform([y_test[i]])[0]
        pred = le.inverse_transform([y_pred[i]])[0]
        print(f"  True: {true:<25} | Pred: {pred}")

    print("\n🎉 Training Complete!")
    return model, le, tfidf, numeric_cols

In [49]:
def load_model_and_predict(items, top_k=5):
    model = joblib.load(MODEL_PATH)
    le = joblib.load(LABEL_ENCODER_PATH)
    tfidf = joblib.load(TFIDF_PATH)
    numeric_cols = joblib.load(os.path.join(MODEL_DIR, "numeric_cols_aurora_v5.joblib"))

    df_pred = pd.DataFrame(items)

    # -----------------------------------------------------
    # 1. Ensure text columns exist and are valid Series
    # -----------------------------------------------------
    if "summary" not in df_pred.columns:
        df_pred["summary"] = ""
    if "description" not in df_pred.columns:
        df_pred["description"] = ""

    df_pred["summary"] = df_pred["summary"].fillna("")
    df_pred["description"] = df_pred["description"].fillna("")

    # -----------------------------------------------------
    # 2. Keyword features
    # -----------------------------------------------------
    for kw in KEYWORDS:
        df_pred[f"kw_{kw}"] = df_pred["summary"].str.contains(kw, case=False, na=False).astype(int)

    df_pred["summary_len"] = df_pred["summary"].apply(lambda x: len(x.split()))

    # -----------------------------------------------------
    # 3. Ensure ALL training numeric columns exist 
    #    (missing → fill with 0)
    # -----------------------------------------------------
    for col in numeric_cols:
        if col not in df_pred.columns:
            df_pred[col] = 0

    # IMPORTANT: enforce same column order
    for col in numeric_cols:
        if col not in df_pred.columns:
            df_pred[col] = 0

    # DO NOT drop the summary column
    X_num = csr_matrix(df_pred[numeric_cols].astype(float).values)


    # Convert dtypes
    df_pred[numeric_cols] = df_pred[numeric_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

    # -----------------------------------------------------
    # 4. TF-IDF
    # -----------------------------------------------------
    X_tfidf_pred = tfidf.transform(df_pred["summary"])

    # -----------------------------------------------------
    # 5. Build numeric sparse matrix
    # -----------------------------------------------------
    X_num_pred = csr_matrix(df_pred[numeric_cols].astype(float).values)

    # Combine sparse tfidf + numeric
    X_sparse_pred = hstack([X_tfidf_pred, X_num_pred]).tocsr()

    # No embeddings (MiniLM disabled)
    X_emb_pred = np.zeros((len(df_pred), 1))

    # Final feature matrix
    X_final_pred = np.hstack([X_sparse_pred.toarray(), X_emb_pred])

    print("\nFinal prediction feature shape:", X_final_pred.shape)

    # -----------------------------------------------------
    # 6. Predict
    # -----------------------------------------------------
    probs = model.predict_proba(X_final_pred)

    # -----------------------------------------------------
    # 7. Pretty-print results
    # -----------------------------------------------------
    print("\n🔮 Prediction Results:")
    for i, row in enumerate(probs):
        print(f"\nItem {i+1}:")
        top_idx = np.argsort(row)[-top_k:][::-1]
        for r, idx in enumerate(top_idx):
            print(f"  Top-{r+1}: {le.inverse_transform([idx])[0]} (prob={row[idx]:.4f})")


In [ ]:
# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    model, le, tfidf, numeric_cols = train_pipeline()

    # Optional quick test
    dummy_items = [
        {
            "summary": "Create responsive UI for login page",
            "issuetype": "Story",
            "status": "To Do",
            "storypoint": 5,
            "priorityid": 3,
            "sprintlength": 14,
            "noofdevelopers": 6,
            "project": "Aurora",
        },
        {
            "summary": "Fix backend API authentication failure",
            "issuetype": "Bug",
            "status": "In Progress",
            "storypoint": 2,
            "priorityid": 1,
            "sprintlength": 10,
            "noofdevelopers": 4,
            "project": "Aurora",
        },
    ]

    load_model_and_predict(dummy_items, top_k=5)


🚀 Training Aurora Assignee Predictor...

--- Loaded Raw Files ---
Issues: (554, 22)
Summary: (568, 10)
Sprints: (40, 22)
Merged: (893, 54)

After whitelist: (824, 54)
assignee
Maxim Khutornenko    263
Bill Farner          160
Kevin Sweeney        114
Mark Chu-Carroll      81
Joshua Cohen          68
Zameer Manji          57
Brian Wickman         54
David McLaughlin      27
Name: count, dtype: int64

After oversampling:
assignee
Bill Farner          263
Brian Wickman        263
David McLaughlin     263
Joshua Cohen         263
Kevin Sweeney        263
Mark Chu-Carroll     263
Maxim Khutornenko    263
Zameer Manji         263
Name: count, dtype: int64

Fitting TF-IDF...
X_final shape: (2104, 641)

Training XGBoost...


C:\Users\risya\AppData\Local\Temp\ipykernel_26356\445099051.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(max_count, replace=True, random_state=42))



📊 Classification Report:
                   precision    recall  f1-score   support

      Bill Farner       0.98      0.98      0.98        52
    Brian Wickman       1.00      1.00      1.00        52
 David McLaughlin       1.00      1.00      1.00        53
     Joshua Cohen       0.98      0.96      0.97        53
    Kevin Sweeney       0.96      0.98      0.97        53
 Mark Chu-Carroll       1.00      1.00      1.00        52
Maxim Khutornenko       0.95      0.98      0.96        53
     Zameer Manji       0.98      0.94      0.96        53

         accuracy                           0.98       421
        macro avg       0.98      0.98      0.98       421
     weighted avg       0.98      0.98      0.98       421


Top-1 Accuracy: 0.981
Top-3 Accuracy: 1.0
Top-5 Accuracy: 1.0

Sample predictions (first 10):
  True: Maxim Khutornenko         | Pred: Maxim Khutornenko
  True: Zameer Manji              | Pred: Zameer Manji
  True: Brian Wickman             | Pred: Brian Wickm